# FINAL PROJECT

## Deliverables:
  1. Your version of this notebook, where you create the required agents (complete all the associated tasks)

  2. After training your agents, you might want to save those learnt values. This can be a dictionary with the `optimal policy`, or `Q-values`. If an approximate method was used, you can provide the `final weights`, etc.

  3. A short report (no more than 10 pages), where you explain your methodology, chosen parameters, visualisations of the learning process (how rewards increase over time, or how loss functions decrease).

  4. Upload to `Faser` a .zip file of all the deliverables (including th optimal Q-values/weights/policies)



## Evaluation


Your projects will be evaluated primarily on the development of the agents, not necessarily in their performance. Being said that, better performance will lead in general to better marks.

## 0. Environment and imports

In [1]:
import os

import numpy as np
import gymnasium as gym
from IPython.display import Image, display
import imageio

from collections import defaultdict
from tqdm import tqdm

from src.logger import LogManager
from src.utils import save_policy, plot_learning_curve
from src.optimization import param_opt_pipeline

In [2]:
manager = LogManager()

main_log = manager.get_logger("Main", "main.log")

In [3]:
env = gym.make("Acrobot-v1", render_mode = "rgb_array")
state = env.reset()
new_step_api = True

main_log.info("--- New Session Started ---")
main_log.info(f"Environment: Acrobot-v1")
main_log.info(f"Initial State: {state}")
main_log.info(f"State Space: {env.observation_space}")
main_log.info(f"Action Space: {env.action_space}")

2026-04-01 02:24:54 | INFO    | --- New Session Started ---
2026-04-01 02:24:54 | INFO    | Environment: Acrobot-v1
2026-04-01 02:24:54 | INFO    | Initial State: (array([ 0.9957951 ,  0.09160866,  0.9998162 , -0.01917309,  0.04707631,
        0.09345089], dtype=float32), {})
2026-04-01 02:24:54 | INFO    | State Space: Box([ -1.        -1.        -1.        -1.       -12.566371 -28.274334], [ 1.        1.        1.        1.       12.566371 28.274334], (6,), float32)
2026-04-01 02:24:54 | INFO    | Action Space: Discrete(3)


## [ACROBOT ENVIRONMENT](https://gymnasium.farama.org/environments/classic_control/acrobot/)

Below is the default discretisation (for the states. Actions are already discrete). If you consider that a different one, could lead to better performance, feel free to modify it, and indicate this in your report.

In [4]:
# --- CONSTANTS ---
BINS = np.array([6, 6, 6, 6, 10, 10])
UPPER_BOUNDS = np.array([1, 1, 1, 1, 4 * np.pi, 9 * np.pi])
LOWER_BOUNDS = np.array([-1, -1, -1, -1, -4 * np.pi, -9 * np.pi])

def discretise(obs):
    """
    Optimized discretisation using NumPy vectorization.
    """
    # Vectorized calculation: (obs - low) / (high - low)
    ratios = (obs - LOWER_BOUNDS) / (UPPER_BOUNDS - LOWER_BOUNDS)
    
    # Scale to bins and clip to stay within array bounds
    new_obs = np.round((BINS - 1) * ratios).astype(int)
    new_obs = np.clip(new_obs, 0, BINS - 1)
    
    return tuple(new_obs)

In [5]:
main_log.info(f"Discretization Bins: {BINS}")

2026-04-01 02:24:54 | INFO    | Discretization Bins: [ 6  6  6  6 10 10]


## Default parameters

I defined the following parameters as the default ones, they will allow you to run things in a reasonable time for testing. However, for properly training your agents, you will need longer time.

**Remark:** the visualisation is just a guidance of performance, the real test of performance is the cumulative rewards.

In [6]:
# --- HYPERPARAMETERS ---
DEFAULT_PARAMS = {
    "alpha": 0.15,
    "gamma": 0.99,
    "epsilon": 1.0,
    "epsilon_decay": 0.999,
    "epsilon_min": 0.05,
    "n_episodes": 500
}

## Useful tools:

You will need to train your algorithms with different sets of parameters. To avoid losing good performing ones, I recommend to use tools such as [Pickle](https://docs.python.org/3/library/pickle.html): This allows to save dictionaries/arrays in a compressed format (using the function `dump`), and then you can load them back, to test the performance (using function `load`)

How to use it? if your trained `Q` value dictionary is called `Q_trained`. You can define the following functions to save/load them.

In [7]:
# error running action = np.argmax(Q[state])
# fixed by changing np.argmax(Q[state]) -> np.argmax(Q[disc_state])

def tester(Q, filename='record.gif', max_steps = 400):
    env = gym.make("Acrobot-v1", render_mode="rgb_array")

    nb_steps = 400
    frames = []
    state, _ = env.reset() # initialize s_0 (a continuous state)
    disc_state = tuple(discretise(state)) # discretise the state and assign it as a tuple (useful for $n$-step SARSA)
    action = np.argmax(Q[disc_state])  # you can use this if you're only saving Q-values

    for k in tqdm(range(nb_steps)):
        frames.append(env.render())

        next_state, reward, done, truncated, _ = env.step(action)
        disc_next_state = tuple(discretise(next_state))  # discretise the next state

        if done or truncated:
            frames.append(env.render())
            break

        action = np.argmax(Q[disc_next_state]) # get the next discrete action, and repeat.
    path = os.path.join("results/", filename)
    imageio.mimsave(path, frames, fps=50)
    return path

In [8]:
def run_experiment(algorithm_func, name, params = DEFAULT_PARAMS):
    """
    Wraps training, plotting, and recording into one call.
    """
    main_log.info(f"Starting Experiment: {name}")
    main_log.info(f"Hyperparameters: {params}")
    
    # 1. Train
    Q_table, rewards = algorithm_func(env, params=params)

    avg_reward = np.mean(rewards[-50:])
    main_log.info(f"Experiment {name} Finished. Final Avg Reward (last 100 eps): {avg_reward:.2f}")
    
    # 2. Save
    save_policy(Q_table, name)
    
    # 3. Visualize
    plot_learning_curve(rewards, name)

    gif_path = tester(Q_table, filename=f"{name.lower()}.gif")
    display(Image(filename=gif_path))
    
    return Q_table, rewards

## Algorithms

In [9]:
def get_epsilon_greedy_action(q_table, state, n_actions, epsilon, seed = 42):
    """Centralized action selection logic."""
    np.random.seed(seed)

    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)
    return np.argmax(q_table[state])

## SARSA implementation

I give you the SARSA implementation for guidance. This should help you to create the algorithms needed for Tasks 1 to 5.

In [10]:
def train_sarsa(env, params = DEFAULT_PARAMS):
    """
    SARSA implementation using a centralized parameter dictionary.
    """
    
    alpha, gamma = params["alpha"], params["gamma"]
    epsilon, epsilon_decay = params["epsilon"], params["epsilon_decay"]
    n_actions = env.action_space.n
    
    Q = defaultdict(lambda: np.zeros(n_actions))
    rewards_history = []

    for ep in tqdm(range(params["n_episodes"]), desc="Training SARSA"):
        obs, _ = env.reset()
        state = discretise(obs)
        action = get_epsilon_greedy_action(Q, state, n_actions, epsilon)

        total_reward = 0
        done = False

        while not done:
            next_obs, reward, term, trunc, _ = env.step(action)
            next_state = discretise(next_obs)
            
            next_action = get_epsilon_greedy_action(Q, next_state, n_actions, epsilon)

            Q[state][action] += alpha * (reward + gamma * Q[next_state][next_action] - Q[state][action])

            state, action = next_state, next_action
            total_reward += reward
            done = term or trunc

        epsilon = max(params["epsilon_min"], epsilon * epsilon_decay)
        rewards_history.append(total_reward)

    return Q, rewards_history

In [11]:
# Q_sarsa, rewards_sarsa = run_experiment(train_sarsa, "SARSA")

## Task 1. Implement Q-learning solution [UG: 30 /  PGT: 20] marks


In here you will need to implement a version of Q-Learning, test and train it, finding the best set of parameters that allows the method to find the solution.

You will need to include:

- implementation of Q-learning
- optimal set of parameters (after trying different combinations)
- visualisations (e.g., execution and/or evolution of rewards)
- Optimal policy and/or Optimal values of Q (e.g., saved as dictionaries )

In [12]:
def train_Q(env, params = DEFAULT_PARAMS, show_progress = False, seed = 42):
    """
    Q-Learning implementation using a centralized parameter dictionary.
    """
    
    alpha, gamma = params["alpha"], params["gamma"]
    epsilon, epsilon_decay = params["epsilon"], params["epsilon_decay"]
    n_actions = env.action_space.n
    
    Q = defaultdict(lambda: np.zeros(n_actions))
    rewards_history = []

    # Use a conditional tqdm
    iterator = range(params["n_episodes"])
    if show_progress:
        iterator = tqdm(iterator, desc="Training Q-Learning", leave=False)

    for ep in iterator:
        obs, _ = env.reset()
        state = discretise(obs)

        total_reward = 0
        done = False

        while not done:
            action = get_epsilon_greedy_action(Q, state, n_actions, epsilon, seed)

            next_obs, reward, term, trunc, _ = env.step(action)
            next_state = discretise(next_obs)

            # Off policy
        
            Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state][action])

            state = next_state
            total_reward += reward
            done = term or trunc

        epsilon = max(params["epsilon_min"], epsilon * epsilon_decay)
        rewards_history.append(total_reward)

    return Q, rewards_history

In [13]:
# Q_learn, rewards_q = run_experiment(train_Q, name = "Q-learning")

### Parameter search

In [14]:
main_log.info("Starting Optuna Hyperparameter Search...")
best_params = DEFAULT_PARAMS.copy()
pareto_trials = param_opt_pipeline(algorithm = train_Q, env = env, n_trials = 100)

2026-04-01 02:24:54 | INFO    | Starting Optuna Hyperparameter Search...


Optimizing train_Q:  31%|███       | 31/100 [48:05<1:47:03, 93.09s/it] 


AssertionError: Should not reach.

In [ ]:
# Iterate to inspect specific trials
for trial in pareto_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")   

Trial #0
  Values: [-49509.2, 280.9508142006355]
  Params: {'alpha': 0.016693285379705526, 'gamma': 0.8261040572984529, 'epsilon_decay': 0.963637104024716}
Trial #3
  Values: [-46196.8, 762.4949573603749]
  Params: {'alpha': 0.2200875934314752, 'gamma': 0.8416666356400487, 'epsilon_decay': 0.9276708180277626}
Trial #4
  Values: [-49880.0, 52.08838642154314]
  Params: {'alpha': 0.003473322654014492, 'gamma': 0.9941980750484857, 'epsilon_decay': 0.9530361838902097}


In [ ]:

# best_params.update(found_params) 

# main_log.info(f"Optimization Complete. Best Params found: {found_params}")

# Q_learn, rewards_q = run_experiment(train_Q, name = "Opt-Q-learning", params = best_params)

## Task 2. Implement n-step SARSA solution [UG: 35 / PGT: 30] marks


In here you will need to implement a version of n-step SARSA, test and train it, finding the best set of parameters that allows the method to find the solution.

You will need to include:

- implementation of n-step SARSA
- optimal set of parameters (after trying different combinations)
- visualisations (e.g., execution and/or evolution of rewards)
- Optimal policy and/or Optimal values of Q (e.g., saved as dictionaries )

In [ ]:
def n_step_sarsa():
    pass

## Task.3 Implement REINFORCE solution [UG: 35 / PGT: 30 ] marks


In here you will need to implement a version of REINFORCE, test and train it, finding the best set of parameters that allows the method to find the solution.

You will need to include:

- implementation of REINFORCE
- optimal set of parameters (after trying different combinations)
- visualisations (e.g., execution and/or evolution of rewards)
- Optimal policy and/or Optimal values of Q (e.g., saved as dictionaries )

In [ ]:
def reinforce():
  pass

## Task.4 ONLY PGT: Implement SARSA($\lambda$) solution [PGT: 20 marks ]


In here you will need to implement a version of SARSA($\lambda$), test and train it, finding the best set of parameters that allows the method to find the solution.

You will need to include:

- implementation of SARSA($\lambda$)
- optimal set of parameters (after trying different combinations)
- visualisations (e.g., execution and/or evolution of rewards)
- Optimal policy and/or Optimal values of Q (e.g., saved as dictionaries )

In [ ]:
def sarsa_lambda():
  pass